## Configuración del comportamiento del modelo

Los modelos de lenguaje no siempre deben responder de la misma manera. En tareas creativas como escribir poesía, queremos respuestas variadas e imaginativas. En tareas técnicas como generar código, preferimos respuestas precisas y consistentes. Los **parámetros de generación** permiten ajustar este comportamiento.

LangChain permite configurar estos parámetros al instanciar el modelo o al realizar cada invocación. ChatOllama hereda los parámetros estándar de LangChain y añade algunos específicos de Ollama.

```mermaid
flowchart TB
    subgraph Parametros["Parámetros de generación"]
        Temp["temperature"]
        NumPred["num_predict"]
        TopP["top_p"]
        TopK["top_k"]
        Repeat["repeat_penalty"]
        Seed["seed"]
    end
    
    Parametros --> Modelo["ChatOllama"]
    Modelo --> Respuesta["Respuesta controlada"]
    
    style Parametros fill:#f59e0b,color:#000
    style Modelo fill:#3b82f6,color:#fff
    style Respuesta fill:#10b981,color:#fff
```

> Los parámetros de generación son opcionales. Si no se especifican, el modelo utiliza valores por defecto que producen un equilibrio entre creatividad y coherencia.

## Temperature: control de la aleatoriedad

El parámetro **temperature** controla cuánta aleatoriedad introduce el modelo al seleccionar las siguientes palabras. Es el parámetro más importante y el más utilizado para ajustar el comportamiento.

* **Temperature baja (0.0 - 0.3)**: Respuestas más deterministas y predecibles
* **Temperature media (0.4 - 0.7)**: Balance entre creatividad y coherencia
* **Temperature alta (0.8 - 1.0+)**: Respuestas más creativas y variadas

In [ ]:
from langchain_ollama import ChatOllama

# Modelo con temperature baja para respuestas consistentes
modelo_preciso = ChatOllama(model="gemma3:1b", temperature=0.1)

# Modelo con temperature alta para respuestas creativas
modelo_creativo = ChatOllama(model="gemma3:1b", temperature=0.9)

### Comparar diferentes temperaturas

Veamos cómo afecta la temperatura a las respuestas del modelo:

In [ ]:
from langchain_ollama import ChatOllama

pregunta = "Escribe una frase sobre el mar"

# Temperature 0 - muy determinista
modelo_frio = ChatOllama(model="gemma3:1b", temperature=0)
print("Temperature 0:")
print(modelo_frio.invoke(pregunta).content)

# Temperature 0.7 - balanceado
modelo_medio = ChatOllama(model="gemma3:1b", temperature=0.7)
print("\nTemperature 0.7:")
print(modelo_medio.invoke(pregunta).content)

# Temperature 1.2 - muy creativo
modelo_caliente = ChatOllama(model="gemma3:1b", temperature=1.2)
print("\nTemperature 1.2:")
print(modelo_caliente.invoke(pregunta).content)

Con temperature 0, el modelo dará la misma respuesta cada vez que ejecutes el código. Con temperature alta, las respuestas variarán significativamente entre ejecuciones.

```mermaid
flowchart LR
    subgraph Rango["Rango de temperature"]
        T0["0.0"]
        T05["0.5"]
        T1["1.0+"]
    end
    
    T0 --> Prec["Preciso y repetible"]
    T05 --> Bal["Balanceado"]
    T1 --> Crea["Creativo y variado"]
    
    style T0 fill:#3b82f6,color:#fff
    style T05 fill:#f59e0b,color:#000
    style T1 fill:#ef4444,color:#fff
```

> Con temperature 0, la respuesta es prácticamente determinista. Para tareas que requieren consistencia (extracción de datos, clasificación), usa temperature baja o cero.

## num_predict: límite de tokens generados

El parámetro **num_predict** limita el número máximo de tokens que el modelo puede generar en su respuesta. Un token es aproximadamente 3/4 de una palabra en español.

In [ ]:
from langchain_ollama import ChatOllama

# Limitar la respuesta a 50 tokens
modelo_corto = ChatOllama(model="gemma3:1b", num_predict=50)

respuesta = modelo_corto.invoke("Explica qué es la inteligencia artificial")
print(respuesta.content)
print(f"\nTokens generados: {respuesta.usage_metadata.get('output_tokens')}")

Este parámetro es útil para:

* **Controlar costes**: Menos tokens generados significa menos recursos consumidos
* **Forzar brevedad**: Obtener respuestas concisas
* **Evitar respuestas infinitas**: Algunos modelos pueden generar texto indefinidamente

In [ ]:
from langchain_ollama import ChatOllama

# Comparar respuestas cortas vs largas
pregunta = "¿Cuáles son los beneficios del ejercicio físico?"

# Respuesta muy corta
modelo_breve = ChatOllama(model="gemma3:1b", num_predict=30)
print("Respuesta breve (30 tokens):")
print(modelo_breve.invoke(pregunta).content)

# Respuesta extendida
modelo_extenso = ChatOllama(model="gemma3:1b", num_predict=200)
print("\nRespuesta extensa (200 tokens):")
print(modelo_extenso.invoke(pregunta).content)

> Si num_predict es demasiado bajo, la respuesta puede cortarse a mitad de frase. Ajusta este valor según la complejidad de la pregunta.

## top_p: muestreo por probabilidad acumulada

El parámetro **top_p** (también llamado *nucleus sampling*) es una estrategia de muestreo que controla la diversidad de las respuestas basándose en la probabilidad acumulada de las palabras candidatas.

Cuando el modelo genera texto, calcula una probabilidad para cada posible palabra siguiente. Con top_p=0.9, el modelo selecciona solo las palabras cuya probabilidad acumulada suma hasta el 90%, descartando las opciones menos probables.

```mermaid
flowchart LR
    subgraph Candidatas[" "]
        direction TB
        P1["'azul' 45%"]
        P2["'celeste' 25%"]
        P3["'claro' 15%"]
        P4["'gris' 8%"]
        P5["'verde' 4%"]
        P6["'rojo' 2%"]
    end
    
    subgraph Acumulado[" "]
        direction TB
        A1["45% ✓"]
        A2["70% ✓"]
        A3["85% ✓"]
        A4["93% ✗"]
    end
    
    subgraph Resultado["top_p=0.9"]
        direction TB
        R1["'azul' ✓"]
        R2["'celeste' ✓"]
        R3["'claro' ✓"]
    end
    
    P1 --> A1 --> R1
    P2 --> A2 --> R2
    P3 --> A3 --> R3
    P4 --> A4
    
    style P1 fill:#10b981,color:#fff
    style P2 fill:#10b981,color:#fff
    style P3 fill:#10b981,color:#fff
    style P4 fill:#f59e0b,color:#000
    style P5 fill:#ef4444,color:#fff
    style P6 fill:#ef4444,color:#fff
    style R1 fill:#10b981,color:#fff
    style R2 fill:#10b981,color:#fff
    style R3 fill:#10b981,color:#fff
```

El diagrama muestra las palabras candidatas ordenadas por probabilidad. Con top_p=0.9, se seleccionan palabras hasta que la probabilidad acumulada alcanza el 90%.

En el ejemplo anterior, para completar "El cielo es de color...", las palabras "azul", "celeste" y "claro" acumulan el 85% de probabilidad. Al añadir "gris" se supera el 90%, por lo que se excluye junto con las palabras menos probables.

In [ ]:
from langchain_ollama import ChatOllama

# top_p bajo - solo las palabras más probables
modelo_conservador = ChatOllama(model="gemma3:1b", top_p=0.5)

# top_p alto - más variedad de palabras
modelo_diverso = ChatOllama(model="gemma3:1b", top_p=0.95)

pregunta = "La capital de España es"

print("top_p=0.5:", modelo_conservador.invoke(pregunta).content)
print("top_p=0.95:", modelo_diverso.invoke(pregunta).content)

Valores típicos de top_p:

* **0.5 - 0.7**: Respuestas más predecibles, solo palabras muy probables
* **0.8 - 0.9**: Balance entre diversidad y coherencia (valor común por defecto)
* **0.95 - 1.0**: Mayor diversidad, incluye palabras menos probables

> El parámetro top_p está disponible en la mayoría de proveedores (OpenAI, Anthropic, Google, Ollama). Es el parámetro de muestreo más universalmente soportado junto con temperature.

## top_k: muestreo por cantidad fija

El parámetro **top_k** es otra estrategia de muestreo que limita la selección a un número fijo de las palabras más probables, independientemente de sus probabilidades.

Con top_k=5, el modelo solo considera las 5 palabras más probables para cada posición, sin importar si la quinta tiene una probabilidad del 10% o del 0.1%.

```mermaid
flowchart LR
    subgraph Candidatas[" "]
        direction TB
        P1["1. 'azul' 45%"]
        P2["2. 'celeste' 25%"]
        P3["3. 'claro' 15%"]
        P4["4. 'gris' 8%"]
        P5["5. 'verde' 4%"]
        P6["6. 'rojo' 2%"]
        P7["7. 'negro' 1%"]
    end
    
    subgraph Filtro["top_k=5"]
        direction TB
        F1["✓ Posición 1-5"]
        F2["✗ Posición 6+"]
    end
    
    subgraph Resultado[" "]
        direction TB
        R1["'azul'"]
        R2["'celeste'"]
        R3["'claro'"]
        R4["'gris'"]
        R5["'verde'"]
    end
    
    P1 --> F1
    P2 --> F1
    P3 --> F1
    P4 --> F1
    P5 --> F1
    P6 --> F2
    P7 --> F2
    
    F1 --> R1
    F1 --> R2
    F1 --> R3
    F1 --> R4
    F1 --> R5
    
    style R1 fill:#10b981,color:#fff
    style R2 fill:#10b981,color:#fff
    style R3 fill:#10b981,color:#fff
    style R4 fill:#10b981,color:#fff
    style R5 fill:#10b981,color:#fff
    style F1 fill:#10b981,color:#fff
    style F2 fill:#ef4444,color:#fff
```

El diagrama muestra cómo top_k selecciona un número fijo de palabras (las 5 primeras), independientemente de sus probabilidades individuales.

In [ ]:
from langchain_ollama import ChatOllama

# top_k muy restrictivo - solo 3 opciones
modelo_restrictivo = ChatOllama(model="gemma3:1b", top_k=3)

# top_k amplio - 50 opciones
modelo_amplio = ChatOllama(model="gemma3:1b", top_k=50)

pregunta = "El cielo es de color"

print("top_k=3:", modelo_restrictivo.invoke(pregunta).content)
print("top_k=50:", modelo_amplio.invoke(pregunta).content)

### Disponibilidad de top_k

A diferencia de top_p, el parámetro **top_k no está disponible en todos los proveedores**. Muchas APIs comerciales no exponen este parámetro:

| Proveedor | temperature | max_tokens | top_p | top_k |
|-----------|-------------|------------|-------|-------|
| OpenAI | ✓ | ✓ | ✓ | ✗ |
| Anthropic | ✓ | ✓ | ✓ | ✓ |
| Google | ✓ | ✓ | ✓ | ✓ |
| Ollama | ✓ | ✓ | ✓ | ✓ |

> OpenAI no permite modificar top_k en su API. Si necesitas control granular sobre el muestreo y trabajas con OpenAI, debes usar únicamente temperature y top_p.

### Diferencia entre top_p y top_k

La diferencia fundamental es cómo determinan el conjunto de palabras candidatas:

* **top_k**: Número fijo de palabras, sin importar sus probabilidades
* **top_p**: Número variable de palabras, basado en probabilidad acumulada

En la práctica, top_p suele producir resultados más consistentes porque se adapta a la distribución de probabilidades. Si una palabra tiene probabilidad del 95%, top_p=0.9 solo seleccionará esa palabra, mientras que top_k=10 aún consideraría 10 opciones.

In [ ]:
from langchain_ollama import ChatOllama

# Combinación de ambos parámetros (cuando están disponibles)
modelo = ChatOllama(
    model="gemma3:1b",
    temperature=0.7,
    top_p=0.9,
    top_k=40
)

respuesta = modelo.invoke("Escribe un título para un artículo sobre Python")
print(respuesta.content)

> Si usas temperature=0, los valores de top_p y top_k no tienen efecto, ya que el modelo siempre seleccionará la palabra más probable.

## repeat_penalty: evitar repeticiones

El parámetro **repeat_penalty** penaliza al modelo por repetir palabras o frases que ya ha generado. Un valor mayor que 1.0 reduce la probabilidad de repeticiones.

In [ ]:
from langchain_ollama import ChatOllama

# Sin penalización por repetición
modelo_sin_penalty = ChatOllama(model="gemma3:1b", repeat_penalty=1.0)

# Con penalización moderada
modelo_con_penalty = ChatOllama(model="gemma3:1b", repeat_penalty=1.2)

pregunta = "Escribe 5 oraciones sobre gatos"

print("Sin penalización:")
print(modelo_sin_penalty.invoke(pregunta).content)

print("\nCon penalización:")
print(modelo_con_penalty.invoke(pregunta).content)

Valores típicos:

* **1.0**: Sin penalización (comportamiento normal)
* **1.1 - 1.2**: Penalización ligera, reduce repeticiones obvias
* **1.3 - 1.5**: Penalización moderada, fuerza más variedad
* **> 1.5**: Penalización fuerte, puede afectar la coherencia

> Un repeat_penalty demasiado alto puede hacer que el modelo evite palabras necesarias y genere texto incoherente. Usa valores entre 1.1 y 1.3 para la mayoría de casos.

## seed: reproducibilidad de resultados

El parámetro **seed** permite obtener resultados reproducibles. Si usas el mismo seed con los mismos parámetros y prompt, el modelo generará la misma respuesta.

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gemma3:1b", temperature=0.7, seed=42)

pregunta = "Inventa un nombre para una mascota"

# Ejecutar varias veces - siempre el mismo resultado
print(modelo.invoke(pregunta).content)
print(modelo.invoke(pregunta).content)
print(modelo.invoke(pregunta).content)

El seed es útil para:

* **Debugging**: Reproducir exactamente el mismo comportamiento
* **Pruebas**: Comparar cambios en prompts con la misma base aleatoria
* **Demostraciones**: Mostrar resultados consistentes

In [ ]:
from langchain_ollama import ChatOllama

# Comparar diferentes seeds
pregunta = "Escribe un verso de 4 palabras"

for seed in [1, 42, 100]:
    modelo = ChatOllama(model="gemma3:1b", temperature=0.8, seed=seed)
    print(f"Seed {seed}: {modelo.invoke(pregunta).content}")

> El seed solo garantiza reproducibilidad con temperature > 0. Con temperature=0, los resultados ya son deterministas sin necesidad de seed.

### La realidad del no-determinismo en LLMs

Aunque el parámetro seed y temperature=0 sugieren que podemos obtener resultados completamente deterministas, la realidad es más compleja. **Incluso con temperature=0 y seed fijo, los LLMs no son verdaderamente deterministas en la práctica**.

¿Por qué ocurre esto? La causa raíz está en la **no-asociatividad de los números de punto flotante** y cómo las GPUs ejecutan operaciones en paralelo:

```
(a + b) + c ≠ a + (b + c)  // En aritmética de punto flotante
```

Cuando una GPU realiza millones de operaciones de suma y multiplicación en paralelo, el orden exacto en que se combinan los resultados puede variar ligeramente entre ejecuciones. Estas pequeñas diferencias numéricas pueden acumularse y, en ciertos puntos críticos, hacer que el modelo seleccione un token diferente.

Un [estudio reciente de Thinking Machines (2025)](https://thinkingmachines.ai/blog/defeating-nondeterminism-in-llm-inference/) demostró este fenómeno: al solicitar 1000 completaciones con temperature=0 usando el mismo prompt, obtuvieron **80 respuestas únicas diferentes**. Las primeras 102 palabras eran idénticas en todas las respuestas, pero a partir del token 103 comenzaban a diverger.

```mermaid
flowchart LR
    subgraph Entrada["Misma entrada"]
        P["Prompt idéntico"]
        T["temperature=0"]
        S["seed=42"]
    end
    
    subgraph GPU["Procesamiento GPU"]
        Op["Operaciones paralelas"]
        FP["Variaciones punto flotante"]
    end
    
    subgraph Salida["Posibles salidas"]
        R1["Respuesta A"]
        R2["Respuesta B"]
        R3["Respuesta C"]
    end
    
    Entrada --> GPU
    GPU --> Salida
    
    style Entrada fill:#3b82f6,color:#fff
    style GPU fill:#f59e0b,color:#000
    style R1 fill:#10b981,color:#fff
    style R2 fill:#10b981,color:#fff
    style R3 fill:#10b981,color:#fff
```

### Implicaciones prácticas

Para la mayoría de aplicaciones, esta variabilidad mínima no es un problema. Sin embargo, es importante tenerla en cuenta:

* **No esperes resultados idénticos** entre diferentes ejecuciones, incluso con los mismos parámetros
* **Los tests automatizados** no deben depender de respuestas exactas del modelo
* **Para debugging**, ejecuta múltiples veces y observa el rango de respuestas posibles
* **En producción**, diseña tu aplicación para tolerar variaciones en las respuestas

> Aunque existen técnicas avanzadas para lograr inferencia determinista (usando kernels especiales "batch-invariant"), estas no están disponibles en las APIs comerciales ni en configuraciones estándar de Ollama.

## Configuración en la invocación

Además de configurar parámetros al instanciar el modelo, puedes modificarlos en cada invocación individual:

In [ ]:
from langchain_ollama import ChatOllama

# Crear modelo con configuración base
modelo = ChatOllama(model="gemma3:1b", temperature=0.5)

# Usar configuración base
respuesta1 = modelo.invoke("Hola")

# Modificar parámetros para esta invocación específica
respuesta2 = modelo.invoke(
    "Escribe algo creativo",
    temperature=0.9,
    num_predict=100
)

Este enfoque permite usar un modelo base y ajustar parámetros según la tarea específica sin crear múltiples instancias.

## Consideraciones para modelos razonadores

Algunos modelos están diseñados específicamente para tareas de razonamiento, como **gpt-oss** (modelos tipo o1/o3), **DeepSeek R1** o modelos con capacidades de *thinking*. Estos modelos razonadores tienen **restricciones importantes** respecto a los parámetros de generación.

### Por qué los modelos razonadores no aceptan ciertos parámetros

Los modelos razonadores utilizan un proceso interno de "pensamiento" (chain-of-thought) que requiere un control preciso sobre la generación de tokens. Introducir aleatoriedad externa mediante temperature o top_p podría:

* Interrumpir la cadena de razonamiento lógico
* Producir pasos de razonamiento incoherentes
* Reducir la calidad de las conclusiones finales

Por esta razón, **la mayoría de modelos razonadores ignoran o rechazan** los parámetros de muestreo estándar.

### Parámetros disponibles según tipo de modelo

```mermaid
flowchart TB
    subgraph Estandar["Modelos estándar"]
        E1["gemma3, llama3, mistral..."]
    end
    
    subgraph ParamsEstandar["Parámetros disponibles"]
        PE1["temperature ✓"]
        PE2["max_tokens ✓"]
        PE3["top_p ✓"]
        PE4["top_k ✓ (según proveedor)"]
        PE5["seed ✓"]
    end
    
    Estandar --> ParamsEstandar
    
    style Estandar fill:#3b82f6,color:#fff
    style PE1 fill:#10b981,color:#fff
    style PE2 fill:#10b981,color:#fff
    style PE3 fill:#10b981,color:#fff
    style PE4 fill:#f59e0b,color:#000
    style PE5 fill:#10b981,color:#fff
```

```mermaid
flowchart TB
    subgraph Razonador["Modelos razonadores"]
        R1["gpt-oss, o1, o3, deepseek-r1..."]
    end
    
    subgraph ParamsRazonador["Parámetros disponibles"]
        PR1["temperature ✗"]
        PR2["max_tokens ✓"]
        PR3["top_p ✗"]
        PR4["top_k ✗"]
        PR5["reasoning_effort ✓"]
    end
    
    Razonador --> ParamsRazonador
    
    style Razonador fill:#f59e0b,color:#000
    style PR1 fill:#ef4444,color:#fff
    style PR2 fill:#10b981,color:#fff
    style PR3 fill:#ef4444,color:#fff
    style PR4 fill:#ef4444,color:#fff
    style PR5 fill:#10b981,color:#fff
```

### El parámetro reasoning_effort

En lugar de controlar la aleatoriedad, los modelos razonadores suelen ofrecer un parámetro **reasoning_effort** que controla cuánto "esfuerzo de pensamiento" dedica el modelo a resolver un problema.

Este parámetro puede expresarse como:

* **Niveles categóricos**: "low", "medium", "high"
* **Presupuesto de tokens**: Número máximo de tokens para el proceso de razonamiento

In [ ]:
from langchain_ollama import ChatOllama

# Con Ollama, algunos modelos razonadores usan configuración interna
modelo_razonador = ChatOllama(model="gpt-oss:20b")

# No especificar temperature, top_p ni top_k
respuesta = modelo_razonador.invoke("¿Cuánto es 15 * 23?")
print(respuesta.content)

### Buenas prácticas con modelos razonadores

1. **No especifiques parámetros de sampling**: El modelo los ignorará o dará error
2. **Usa max_tokens/num_predict**: Para limitar la longitud total de la respuesta
3. **Configura reasoning_effort si está disponible**: Ajusta según la complejidad de la tarea
4. **Acepta respuestas más largas**: El razonamiento visible consume tokens adicionales

In [ ]:
from langchain_ollama import ChatOllama

# Configuración correcta para modelos razonadores en Ollama
modelo_razonador = ChatOllama(
    model="gpt-oss:20b",
    num_predict=1000  # Espacio suficiente para razonamiento + respuesta
)

problema = """
Resuelve paso a paso: Si Ana tiene 3 manzanas más que el doble 
de las que tiene Pedro, y entre los dos tienen 15 manzanas, 
¿cuántas tiene cada uno?
"""

respuesta = modelo_razonador.invoke(problema)
print(respuesta.content)

### Resumen de compatibilidad por tipo de modelo

| Parámetro | Modelos estándar | Modelos razonadores |
|-----------|------------------|---------------------|
| temperature | ✓ Soportado | ✗ Ignorado/Error |
| max_tokens / num_predict | ✓ Soportado | ✓ Soportado |
| top_p | ✓ Soportado | ✗ Ignorado/Error |
| top_k | Según proveedor | ✗ Ignorado/Error |
| seed | ✓ Soportado | Según modelo |
| reasoning_effort | ✗ No aplica | ✓ Cuando disponible |

> Consulta siempre la documentación del modelo y proveedor específico. Las capacidades pueden variar entre versiones y las APIs evolucionan constantemente.

## Casos de uso y configuraciones recomendadas

### Generación de código

Para código, queremos respuestas precisas y consistentes:

In [ ]:
from langchain_ollama import ChatOllama

modelo_codigo = ChatOllama(
    model="gemma3:1b",
    temperature=0.1,
    repeat_penalty=1.0  # No penalizar repeticiones en código
)

respuesta = modelo_codigo.invoke("Escribe una función Python que calcule el factorial")
print(respuesta.content)

### Escritura creativa

Para textos creativos, más aleatoriedad y variedad:

In [ ]:
from langchain_ollama import ChatOllama

modelo_creativo = ChatOllama(
    model="gemma3:1b",
    temperature=0.9,
    top_p=0.95,
    repeat_penalty=1.2
)

respuesta = modelo_creativo.invoke("Escribe el inicio de un cuento de ciencia ficción")
print(respuesta.content)

### Extracción de información

Para extraer datos estructurados, máxima precisión:

In [ ]:
from langchain_ollama import ChatOllama

modelo_extraccion = ChatOllama(
    model="gemma3:1b",
    temperature=0,
    num_predict=100
)

texto = "Juan García, nacido el 15 de marzo de 1990, trabaja como ingeniero en Madrid."
prompt = f"Extrae el nombre, fecha de nacimiento, profesión y ciudad del siguiente texto: {texto}"

respuesta = modelo_extraccion.invoke(prompt)
print(respuesta.content)

### Resumen de configuraciones recomendadas

| Caso de uso | temperature | top_p | max_tokens |
|-------------|-------------|-------|------------|
| Código | 0 - 0.2 | 0.9 | Variable |
| Escritura creativa | 0.8 - 1.0 | 0.95 | Variable |
| Extracción de datos | 0 | 0.9 | Bajo |
| Chat general | 0.5 - 0.7 | 0.9 | Variable |
| Razonamiento | No usar | No usar | Alto (incluye pensamiento) |

> Esta tabla usa los parámetros más universalmente soportados. Si trabajas exclusivamente con Ollama, también puedes ajustar top_k y repeat_penalty para un control más fino.

Con estos parámetros puedes ajustar el comportamiento de los modelos para adaptarlos a diferentes tareas y obtener resultados óptimos en cada situación.

